# Asisten AI Legal berbasis RAG — Regulasi Cipta Kerja

Notebook ini membangun **pipeline Retrieval-Augmented Generation (RAG)** modular untuk
asisten hukum tim legal internal. Sistem menjawab pertanyaan seputar **empat regulasi
turunan Undang-Undang Cipta Kerja** dengan jawaban yang **di-grounding ke pasal sumber**,
bukan halusinasi model.

**Empat dokumen sumber pengetahuan:**

| Kode | Regulasi | Tentang |
|---|---|---|
| `PP_5_2021` | PP No. 5 Tahun 2021 | Perizinan Berusaha Berbasis Risiko |
| `PP_35_2021` | PP No. 35 Tahun 2021 | PKWT, Alih Daya, Waktu Kerja, PHK |
| `PP_51_2023` | PP No. 51 Tahun 2023 | Pengupahan (formula upah minimum) |
| `UU_6_2023` | UU No. 6 Tahun 2023 | Cipta Kerja (omnibus, 15 BAB) |

**Desain kunci:** pengetahuan hukum berasal dari **retrieval** (4 PDF), bukan dari bobot model.
Model generator (`Llama-3.2-3B` yang telah di-fine-tune) berperan menyusun jawaban Bahasa
Indonesia formal berdasarkan konteks yang di-retrieve.

**Arsitektur modular (satu layer per tanggung jawab, mudah di-swap ke domain lain):**

```
PDF → Loader → Cleaner → Chunker → Embedder (BGE-M3) → Vector Store (ChromaDB)
    → Retriever → Generator (fine-tuned Llama)
```

Bagian **A (RAG Dasar)** di notebook ini mencakup pipeline end-to-end pertama: memuat 4 PDF,
memecah menjadi chunk per-pasal dengan *overlap* eksplisit, meng-*embed* dengan BGE-M3,
menyimpan ke ChromaDB persisten (18 koleksi), lalu men-*generate* jawaban dengan model
fine-tuned.


## A.1 — Lingkungan & Dependensi

Jalankan sel instalasi di bawah. **Pada eksekusi pertama**, setelah instalasi selesai,
lakukan *Restart runtime* (Runtime → Restart session) lalu **Run All** ulang agar
seluruh library ter-*update* termuat bersih. Notebook dirancang untuk **Google Colab
GPU T4**.

In [1]:
# Instalasi dependensi RAG (embedder, vector store, retriever, generator)
%pip install -q chromadb sentence-transformers pypdf pyyaml rank-bm25 duckduckgo-search
%pip install -q -U transformers accelerate bitsandbytes
print(">> Dependensi terpasang. Jika ini eksekusi pertama: Restart runtime lalu Run All ulang.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/6

In [2]:
# Import inti + reproducibility (seed=42 di semua operasi acak)
import os, re, json, random, time
from pathlib import Path
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__, "| CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128 | CUDA tersedia: True
GPU: Tesla T4


## A.2 — Konfigurasi (config-driven)

Seluruh *magic number* (ukuran chunk, overlap, top-k, ID model, ambang) dikumpulkan dalam
satu objek konfigurasi. Pendekatan ini membuat pipeline **portabel ke domain lain** —
cukup ganti PDF sumber dan nilai konfigurasi, tanpa menyentuh logika.

In [3]:
# Konfigurasi terpusat untuk pipeline RAG
RAG_CONFIG = {
    "seed": 42,
    "chunker": {
        "chunk_size": 1200,      # target char per chunk (batas rubric <= 5000)
        "chunk_overlap": 100,    # overlap EKSPLISIT antar sub-chunk (bukan default library)
        "hard_cap": 5000,        # batas keras char per chunk
    },
    "embedder": {
        "model_id": "BAAI/bge-m3",
        "batch_size": 32,
        "normalize": True,
    },
    "vector_store": {
        "distance_metric": "cosine",
    },
    "retriever": {
        "top_k": 5,              # jumlah chunk konteks untuk generator (tier Basic)
    },
    "generator": {
        "model_id": "nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO",  # hasil fine-tuning K1
        "max_new_tokens": 512,
        "temperature": 0.3,      # rendah -> jawaban legal deterministik
        "top_p": 0.9,
        "repetition_penalty": 1.1,
    },
    "system_prompt": (
        "Anda adalah asisten hukum untuk tim legal internal. Jawab HANYA berdasarkan "
        "konteks pasal yang diberikan. Gunakan Bahasa Indonesia hukum yang formal dan "
        "ringkas. Jika konteks tidak memuat jawaban, katakan: 'Informasi tidak ditemukan "
        "pada regulasi yang tersedia.'"
    ),
    "context_template": "Konteks pasal:\n{sources}\n\nPertanyaan: {question}\nJawaban:",
}
print("Konfigurasi RAG dimuat.")

Konfigurasi RAG dimuat.


## A.3 — Mount Google Drive & Path

Empat PDF sumber diletakkan di Drive: `/content/drive/MyDrive/PGABL/data/raw/`.
ChromaDB persisten disimpan ke Drive agar indeks bertahan lintas sesi.

In [4]:
# Mount Drive + definisi path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/PGABL")
except ImportError:
    BASE = Path("./PGABL")   # fallback lokal

RAW_DIR = BASE / "data" / "raw"
CHROMA_DIR = str(BASE / "chroma_db")
(BASE / "data").mkdir(parents=True, exist_ok=True)

# Empat PDF: nama file -> kode sumber (path-safe)
PDF_FILES = {
    "PP_5_2021":  RAW_DIR / "PP_5_2021.pdf",
    "PP_35_2021": RAW_DIR / "PP_35_2021.pdf",
    "PP_51_2023": RAW_DIR / "PP_51_2023.pdf",
    "UU_6_2023":  RAW_DIR / "UU_6_2023.pdf",
}
for k, p in PDF_FILES.items():
    print(f"{k:12s}: {'OK ' if p.exists() else 'HILANG'} {p}")
assert all(p.exists() for p in PDF_FILES.values()), (
    "Sebagian PDF belum ada. Unggah 4 PDF ke /content/drive/MyDrive/PGABL/data/raw/ "
    "dengan nama: PP_5_2021.pdf, PP_35_2021.pdf, PP_51_2023.pdf, UU_6_2023.pdf")

Mounted at /content/drive
PP_5_2021   : OK  /content/drive/MyDrive/PGABL/data/raw/PP_5_2021.pdf
PP_35_2021  : OK  /content/drive/MyDrive/PGABL/data/raw/PP_35_2021.pdf
PP_51_2023  : OK  /content/drive/MyDrive/PGABL/data/raw/PP_51_2023.pdf
UU_6_2023   : OK  /content/drive/MyDrive/PGABL/data/raw/UU_6_2023.pdf


## A.4 — Loader, Cleaner, dan Chunker

Tiga layer preprocessing (di-*inline* agar notebook mandiri):

- **Loader** — ekstrak teks per-halaman dengan `pypdf`.
- **Cleaner** — buang header/footer berulang (`PRESIDEN REPUBLIK INDONESIA`, nomor halaman),
  normalisasi artefak OCR (`2O21`→`2021`), rekonstruksi kata terpotong antar-halaman.
- **Chunker** — pecah **per-pasal**; satu pasal = satu chunk bila ≤ `chunk_size`, dan
  **pasal panjang di-sub-split dengan overlap eksplisit 100 karakter**. Untuk `UU_6_2023`,
  setiap chunk dipetakan ke salah satu dari **15 klaster (= 15 BAB omnibus)**; bagian
  Penjelasan di-route ke klaster berdasarkan konten.

In [5]:
# --- Loader ---
"""
PGABL - PDF Loader untuk RAG (Tahap 1b + Tahap 3).

Design:
- Default pakai pypdf (fastest, identical output vs pdfplumber untuk 4 PDF ini).
- Fallback pdfplumber tersedia via load_pdf(strategy="pdfplumber").
- Reusable ke domain lain: ganti file di data/raw/, tidak sentuh code.

Verify-first prototype: scripts/01_verify_pdf_loader.py
Hasil: semua 4 PDF (PP_5_2021, PP_35_2021, PP_51_2023, UU_6_2023) pakai pypdf clean.
"""

from __future__ import annotations
from pathlib import Path
from typing import Literal


LoaderStrategy = Literal["pypdf", "pdfplumber"]


def load_pdf_pages(pdf_path: Path, strategy: LoaderStrategy = "pypdf") -> list[str]:
    """
    Load semua halaman PDF menjadi list string per-halaman.

    Args:
        pdf_path: absolute atau relative path ke PDF
        strategy: "pypdf" (default, 2-4x lebih cepat) atau "pdfplumber" (robust untuk edge cases)

    Returns:
        List of strings, satu per halaman. String kosong kalau extract_text gagal.
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    if strategy == "pypdf":
        return _load_with_pypdf(pdf_path)
    elif strategy == "pdfplumber":
        return _load_with_pdfplumber(pdf_path)
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Use 'pypdf' or 'pdfplumber'.")


def _load_with_pypdf(pdf_path: Path) -> list[str]:
    import pypdf
    pages: list[str] = []
    with open(pdf_path, "rb") as f:
        reader = pypdf.PdfReader(f)
        for page in reader.pages:
            try:
                pages.append(page.extract_text() or "")
            except Exception as e:
                pages.append(f"[pypdf error: {e}]")
    return pages


def _load_with_pdfplumber(pdf_path: Path) -> list[str]:
    import pdfplumber
    pages: list[str] = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            try:
                pages.append(page.extract_text() or "")
            except Exception as e:
                pages.append(f"[pdfplumber error: {e}]")
    return pages

In [6]:
# --- Cleaner ---
"""
PGABL - Text Cleaner untuk PDF Legal Indonesia (Tahap 1b).

Kegunaan: strip repeated header/footer + normalize OCR artefak yang common
di dokumen regulasi Indonesia (PP, UU).

Design:
- Config regex di atas (HEADER_PATTERNS, FOOTER_PATTERNS, OCR_REPLACEMENTS).
- Ganti pattern untuk domain lain via extend config, jangan ubah logic.

Verify-first prototype: scripts/02_verify_chunker.py
Hasil: 4 PDF dgn cleanup delta -37 (PP_5_2021), -2052 (PP_35_2021),
       -590 (PP_51_2023), -23,746 (UU_6_2023).
"""

from __future__ import annotations
import re


# ==================== Header patterns (repeat tiap halaman) ====================
HEADER_PATTERNS = [
    # PRESIDEN\nREPUBLIK INDONESIA (dan variant garbled: NEPUBUK, REPUBLTI(, REPUELIK, dst)
    r"PRESIDEN\s*\n\s*(?:REPUBLIK|NEPUBUK|REPUBLTI\(?|REPUBLTK|REPTIBLIK|REPUELIK)\s*INDONESIA\s*\n",
    # LEMBARAN NEGARA header (cover page only)
    r"LEMBARAN\s+NEGARA\s*\n\s*REPUBLIK\s+INDONESIA\s*\n",
    # SALINAN header (tepat sebelum PRESIDEN)
    r"^\s*SALINAN\s*\n",
]


# ==================== Footer patterns ====================
FOOTER_PATTERNS = [
    # Page number "-N-" (dgn spasi/tanpa)
    r"\n\s*-\s*\d+\s*-\s*\n",
    r"\n\s*-\s*\d+\s*-\s*$",
    # SK No XXXXX A footer (tapi bukan SK dalam text lain)
    r"SK\s+No\s*[\d\s]+\s*[A-Z]?\s*(?:\n|$)",
]


# ==================== OCR replacements ====================
# Format: {regex_pattern: replacement}
OCR_REPLACEMENTS = {
    # Garbled REPUBLIK variants
    r"\bNEPUBUK\b": "REPUBLIK",
    r"\bREPUBLTI\(?\b": "REPUBLIK",
    r"\bREPUBLTK\b": "REPUBLIK",
    r"\bREPTIBLIK\b": "REPUBLIK",
    r"\bREPUELIK\b": "REPUBLIK",
    # Lower-case 'l' bukannya 'I'
    r"\blndonesia\b": "Indonesia",
    r"\blndo\b": "Indo",
    # Common misspellings from OCR
    r"\bkemanusraan\b": "kemanusiaan",
    r"\bpersekutrran\b": "persekutuan",
}


def normalize_ocr(text: str) -> str:
    """Fix OCR artefak yang common di dokumen legal Indonesia."""
    # Digit O di dalam angka (2O21 -> 2021), tapi bukan huruf O di kata biasa
    text = re.sub(r"(\d)O(\d)", r"\g<1>0\g<2>", text)
    text = re.sub(r"(\d)O$", r"\g<1>0", text, flags=re.MULTILINE)
    # Digit l di dalam angka (l945 -> 1945)
    text = re.sub(r"(\d)l(\d)", r"\g<1>1\g<2>", text)
    text = re.sub(r"\bl(\d{3,4})\b", r"1\g<1>", text)
    # Word-level replacements
    for pattern, repl in OCR_REPLACEMENTS.items():
        text = re.sub(pattern, repl, text)
    return text


def strip_headers_footers(text: str) -> str:
    """Buang repeated header/footer patterns."""
    for pat in HEADER_PATTERNS:
        text = re.sub(pat, "", text, flags=re.MULTILINE | re.IGNORECASE)
    for pat in FOOTER_PATTERNS:
        text = re.sub(pat, "\n", text, flags=re.MULTILINE)
    return text


def reconstruct_hyphenated_words(pages: list[str]) -> list[str]:
    """
    Gabungkan kata yang terpotong antar halaman (mis. 'keman-\nusraan' -> 'kemanusiaan').
    In-place modification, tapi return new list.
    """
    result = list(pages)
    for i in range(len(result) - 1):
        current = result[i].rstrip()
        m = re.search(r"(\w+)-\s*$", current)
        if m:
            trailing = m.group(1)
            next_page = result[i + 1].lstrip()
            n = re.match(r"(\w+)(.*)", next_page, re.DOTALL)
            if n:
                first_word = n.group(1)
                rest = n.group(2)
                result[i] = current[: -(len(trailing) + 1)] + trailing + first_word
                result[i + 1] = rest
    return result


def clean_pages(pages: list[str]) -> list[str]:
    """
    Full pipeline: reconstruct hyphenated -> strip headers/footers -> normalize OCR.

    Return: list halaman baru dgn text bersih.
    """
    pages = reconstruct_hyphenated_words(pages)
    cleaned = []
    for p in pages:
        p = strip_headers_footers(p)
        p = normalize_ocr(p)
        cleaned.append(p)
    return cleaned

In [7]:
# --- Chunker (per-pasal + sub-split overlap + routing klaster UU) ---
"""
PGABL - Chunker untuk Legal PDF (Tahap 1b + Tahap 3).

Strategy per-pasal (flat) untuk semua 4 PDF:
- Regex `^\\s*Pasal\\s+(\\d+)\\s*$` split jadi 1 blok per pasal.
- Blok pasal > chunk_size di-sub-split dgn OVERLAP EKSPLISIT (rubric WAJIB).
- Metadata: bab (Roman numeral), pasal, chunk_id, pdf_source.
- Khusus UU_6_2023: map BAB Roman I-XV -> klaster_id 1-15 + label tema
  (dipakai untuk 15 collection ChromaDB per klaster di Tahap 3).

Verify-first prototype: scripts/09_full_ingest_rag.py -> outputs/samples/.

Reusable ke domain lain: ganti PASAL_REGEX / BAB_REGEX / klaster map via argumen,
logic split tidak berubah.
"""

from __future__ import annotations
import re
from typing import Optional


# Pasal detection - baris tersendiri, toleran spasi
PASAL_REGEX = re.compile(r"^\s*Pasal\s+(\d+)\s*$", re.MULTILINE | re.IGNORECASE)

# BAB detection - Roman numeral standalone
BAB_REGEX = re.compile(r"^\s*BAB\s+([IVXLCDM]+)\s*$", re.MULTILINE)

# "Cukup jelas" di section Penjelasan (low-signal, di-skip)
CUKUP_JELAS_REGEX = re.compile(
    r"Pasal\s+\d+\s*\n\s*Cukup\s+jelas\s*\.?\s*(?:\n|$)",
    re.IGNORECASE,
)


# ==================== UU 6/2023: BAB (Roman) -> Klaster ====================
# UU 6/2023 (Cipta Kerja) punya TEPAT 15 BAB. "15 klaster" di desain = 15 BAB.
# Terverifikasi via recon lokal (semua BAB I-XV terdeteksi bersih).
# Label tema dipakai untuk sitasi sumber yang enak dibaca legal team.
UU_6_2023_KLASTER_LABELS = {
    1: "Ketentuan Umum",
    2: "Asas, Tujuan, dan Ruang Lingkup",
    3: "Peningkatan Ekosistem Investasi dan Kegiatan Berusaha",
    4: "Ketenagakerjaan",
    5: "Koperasi dan UMK-M",
    6: "Kemudahan Berusaha",
    7: "Dukungan Riset dan Inovasi",
    8: "Pengadaan Tanah",
    9: "Kawasan Ekonomi",
    10: "Investasi Pemerintah Pusat dan Proyek Strategis Nasional",
    11: "Administrasi Pemerintahan",
    12: "Pengawasan dan Pembinaan",
    13: "Ketentuan Lain-Lain",
    14: "Ketentuan Peralihan",
    15: "Ketentuan Penutup",
}

_ROMAN_MAP = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}


# Keyword routing untuk chunk PENJELASAN (yang tak bisa di-track per-pasal karena didominasi
# referensi UU sektor). Chunk penjelasan diarahkan ke klaster dgn hit keyword terbanyak;
# kalau nol match -> fallback klaster 3 (Ekosistem Investasi = cluster dominan/catch-all).
# Keyword lowercase; dicek substring pada text lowercase.
UU_6_2023_KLASTER_KEYWORDS = {
    4: ["pekerja", "buruh", "pengupahan", "upah minimum", "upah kerja", "lembur",
        "pkwt", "pemutusan hubungan kerja", "pesangon", "tenaga kerja", "serikat pekerja",
        "waktu kerja", "hubungan kerja", "perjanjian kerja", "jaminan sosial",
        "tenaga kerja asing", "pelatihan kerja"],
    5: ["koperasi", "usaha mikro", "usaha kecil", "usaha menengah", "umk-m", "umkm"],
    6: ["perseroan terbatas", "badan hukum", "keimigrasian", "hak kekayaan intelektual",
        "paten", "merek", "perkumpulan"],
    8: ["pengadaan tanah", "ganti kerugian", "pelepasan hak", "bank tanah"],
    9: ["kawasan ekonomi khusus", "kawasan perdagangan bebas", "kawasan ekonomi",
        "zona ekonomi"],
    10: ["proyek strategis nasional", "investasi pemerintah pusat", "lembaga pengelola investasi"],
    7: ["riset dan inovasi", "riset", "inovasi"],
    3: ["perizinan berusaha", "nomor induk berusaha", "sistem oss", "kbli", "berbasis risiko",
        "lingkungan hidup", "bangunan gedung", "tata ruang", "kehutanan", "sumber daya air",
        "kelautan", "perikanan", "ketenaganukliran", "pelayaran", "penerbangan", "pos",
        "telekomunikasi", "energi", "ketenagalistrikan", "perdagangan", "perindustrian"],
}


def route_penjelasan_by_keyword(text: str, fallback_klaster: int = 3) -> int:
    """Klaster untuk 1 chunk penjelasan via hit keyword terbanyak; nol match -> fallback."""
    low = text.lower()
    best_k, best_hits = fallback_klaster, 0
    for kid, kws in UU_6_2023_KLASTER_KEYWORDS.items():
        hits = sum(low.count(kw) for kw in kws)
        if hits > best_hits:
            best_k, best_hits = kid, hits
    return best_k


def roman_to_int(roman: str) -> Optional[int]:
    """Konversi Roman numeral ke int. Return None kalau input invalid."""
    if not roman:
        return None
    roman = roman.upper()
    total, prev = 0, 0
    for ch in reversed(roman):
        if ch not in _ROMAN_MAP:
            return None
        val = _ROMAN_MAP[ch]
        total += -val if val < prev else val
        prev = max(prev, val)
    return total if total > 0 else None


def find_pasal_boundaries(full_text: str) -> list[tuple[int, str]]:
    """Return list of (char_offset, pasal_number)."""
    return [(m.start(), m.group(1)) for m in PASAL_REGEX.finditer(full_text)]


def find_bab_boundaries(full_text: str) -> list[tuple[int, str]]:
    """Return list of (char_offset, bab_id)."""
    return [(m.start(), m.group(1)) for m in BAB_REGEX.finditer(full_text)]


def which_bab_for_offset(offset: int, bab_boundaries: list[tuple[int, str]]) -> Optional[str]:
    """Return BAB yang mencakup offset ini, atau None kalau di luar BAB apapun."""
    current = None
    for pos, bab in bab_boundaries:
        if pos <= offset:
            current = bab
        else:
            break
    return current


def split_text_with_overlap(text: str, chunk_size: int, overlap: int) -> list[str]:
    """
    Sliding-window split dgn OVERLAP EKSPLISIT (rubric WAJIB, bukan default library).

    Kalau text <= chunk_size -> 1 chunk utuh (tidak dipecah).
    Kalau lebih panjang -> potongan sepanjang chunk_size dgn step (chunk_size - overlap).
    Invariant: part[k][-overlap:] == part[k+1][:overlap] (overlap persis `overlap` char).
    """
    if overlap >= chunk_size:
        raise ValueError(f"overlap ({overlap}) harus < chunk_size ({chunk_size})")
    if len(text) <= chunk_size:
        return [text]
    parts: list[str] = []
    step = chunk_size - overlap
    start = 0
    while start < len(text):
        parts.append(text[start:start + chunk_size])
        if start + chunk_size >= len(text):
            break
        start += step
    return parts


def chunk_by_pasal(
    full_text: str,
    pdf_source: str,
    include_bab_metadata: bool = True,
    chunk_size: int = 1200,
    chunk_overlap: int = 100,
    hard_cap: int = 5000,
    skip_cukup_jelas: bool = True,
    map_klaster_by_bab: bool = False,
) -> list[dict]:
    """
    Chunker flat per-pasal dgn sub-split overlap eksplisit.

    Args:
        full_text: teks lengkap PDF (setelah cleaner).
        pdf_source: nama PDF tanpa .pdf (mis. "PP_35_2021").
        include_bab_metadata: kalau True, tambah field 'bab' di setiap chunk.
        chunk_size: panjang target chunk (char). Blok pasal > ini di-sub-split.
        chunk_overlap: overlap antar sub-chunk (char). EKSPLISIT (rubric WAJIB).
        hard_cap: batas keras char per chunk (rubric max 5000) - safety net.
        skip_cukup_jelas: skip pasal Penjelasan yang isinya cuma "Cukup jelas".
        map_klaster_by_bab: kalau True (khusus UU_6_2023), tambah klaster_id + klaster_label
                            dari BAB Roman (I->1 .. XV->15).

    Returns:
        List dict, tiap chunk punya:
          - chunk_id: str unik (mis. "PP_35_2021_pasal_1_0" atau "..._0_sub1" utk sub-chunk)
          - pdf_source, pasal, bab (Optional)
          - text: str (isi, di-cap di hard_cap)
          - length: int (panjang blok pasal ORIGINAL, sebelum sub-split)
          - char_len: int (panjang text chunk ini)
          - is_subchunk: bool, sub_index: int, n_subchunks: int
          - (UU only) klaster_id: Optional[int], klaster_label: Optional[str]
    """
    pasal_boundaries = find_pasal_boundaries(full_text)
    bab_boundaries = find_bab_boundaries(full_text) if include_bab_metadata else []

    chunks: list[dict] = []
    for i, (start, pasal_num) in enumerate(pasal_boundaries):
        end = pasal_boundaries[i + 1][0] if i + 1 < len(pasal_boundaries) else len(full_text)
        block = full_text[start:end].strip()

        # Skip "Cukup jelas" pasal (low-signal dari Penjelasan)
        if skip_cukup_jelas and CUKUP_JELAS_REGEX.search(block) and len(block) < 100:
            continue

        bab = which_bab_for_offset(start, bab_boundaries) if include_bab_metadata else None
        klaster_id = None
        klaster_label = None
        if map_klaster_by_bab and bab is not None:
            kid = roman_to_int(bab)
            if kid is not None and kid in UU_6_2023_KLASTER_LABELS:
                klaster_id = kid
                klaster_label = UU_6_2023_KLASTER_LABELS[kid]

        sub_texts = split_text_with_overlap(block, chunk_size, chunk_overlap)
        n_sub = len(sub_texts)
        for j, sub in enumerate(sub_texts):
            chunk_id = (
                f"{pdf_source}_pasal_{pasal_num}_{i}"
                if n_sub == 1
                else f"{pdf_source}_pasal_{pasal_num}_{i}_sub{j}"
            )
            chunk = {
                "chunk_id": chunk_id,
                "pdf_source": pdf_source,
                "pasal": pasal_num,
                "bab": bab,
                "text": sub[:hard_cap],
                "length": len(block),
                "char_len": len(sub[:hard_cap]),
                "char_offset": start,          # offset blok pasal di full_text (utk routing UU)
                "is_subchunk": n_sub > 1,
                "sub_index": j,
                "n_subchunks": n_sub,
            }
            if map_klaster_by_bab:
                chunk["klaster_id"] = klaster_id
                chunk["klaster_label"] = klaster_label
                chunk["jenis"] = "batang_tubuh"   # default; di-refine oleh refine_uu_klaster
            chunks.append(chunk)

    return chunks


# ==================== UU 6/2023: routing Penjelasan -> klaster benar ====================
# Masalah: PENJELASAN besar (pasal-demi-pasal) ada SETELAH BAB XV & tak pakai header BAB,
# jadi kalau naif semua chunk Penjelasan mewarisi bab=XV -> nyasar ke klaster_15.
# Fix: (1) deteksi batas Penjelasan, (2) bangun backbone pasal-Cipta-Kerja (1..186, berurutan)
#      dari batang tubuh -> peta own_pasal->klaster, (3) route tiap chunk Penjelasan via
#      pasal yang dijelaskannya (tracking monotonic; referensi UU sektor = noise, di-skip).

def find_penjelasan_offset(full_text: str) -> int:
    """Offset awal PENJELASAN besar = 'PENJELASAN ATAS' pertama SETELAH BAB terakhir."""
    bab_bounds = find_bab_boundaries(full_text)
    last_bab_off = bab_bounds[-1][0] if bab_bounds else 0
    for m in re.finditer(r"PENJELASAN\s*\n?\s*ATAS", full_text):
        if m.start() > last_bab_off:
            return m.start()
    ms = [m.start() for m in re.finditer(r"\bPENJELASAN\b", full_text)]
    return ms[-1] if ms else len(full_text)


def build_own_pasal_backbone(full_text: str, pen_offset: int) -> list[tuple[int, int]]:
    """
    Peta breakpoint (own_pasal_start, klaster_id) per BAB di batang tubuh.

    Robust: dijangkar ke 15 header BAB (terdeteksi bersih), BUKAN rantai berurutan yang
    rapuh terhadap glitch OCR. Untuk tiap BAB (dedupe first-occurrence, buang stray duplikat),
    ambil 'Pasal N' standalone PERTAMA setelah header BAB itu sebagai own_pasal awal klaster.
    Guard monotonic (num > prev) supaya stray/embedded tidak merusak urutan.

    Return list (own_pasal_start, klaster_id) urut naik own_pasal.
    Contoh: [(1,1),(2,2),(4,3),(80,4),(85,5),...] -> klaster_for_own_pasal binary-lookup.
    """
    matches = list(PASAL_REGEX.finditer(full_text))
    # Dedupe BAB: keep first occurrence per roman (buang stray duplikat spt 'BAB V' kedua)
    seen: set[str] = set()
    ordered_bab: list[tuple[int, str]] = []
    for off, roman in find_bab_boundaries(full_text):
        if off >= pen_offset:
            break
        if roman not in seen:
            seen.add(roman)
            ordered_bab.append((off, roman))

    backbone: list[tuple[int, int]] = []
    prev = -1
    for off, roman in ordered_bab:
        kid = roman_to_int(roman)
        if kid is None:
            continue
        for m in matches:
            if m.start() > off:
                num = int(m.group(1))
                if prev < num <= 250:          # monotonic + plausible
                    backbone.append((num, kid))
                    prev = num
                break
    return backbone


def klaster_for_own_pasal(n: int, backbone: list[tuple[int, int]]) -> Optional[int]:
    """Klaster dari own_pasal terbesar di backbone yang <= n."""
    kid = None
    for own, k in backbone:
        if own <= n:
            kid = k
        else:
            break
    return kid


def refine_uu_klaster(chunks: list[dict], full_text: str) -> dict:
    """
    Refine klaster UU 6/2023 in-place:
      - tandai jenis (batang_tubuh / penjelasan) via offset batas Penjelasan
      - batang tubuh: pakai klaster dari BAB (sudah ada); bab=None -> klaster_1 (front matter)
      - penjelasan: route ke klaster benar via own-pasal yang dijelaskan (tracking monotonic)
    Return ringkasan (offset, ukuran backbone) untuk laporan verifikasi.
    """
    pen_offset = find_penjelasan_offset(full_text)
    backbone = build_own_pasal_backbone(full_text, pen_offset)

    # Batang tubuh: klaster dari BAB (sudah di-set). Penjelasan: keyword routing (per-pasal
    # tidak reliable karena didominasi referensi UU sektor setelah own Pasal ~17).
    n_penjelasan = 0
    for c in chunks:
        off = c.get("char_offset", 0)
        if off >= pen_offset:
            c["jenis"] = "penjelasan"
            k = route_penjelasan_by_keyword(c["text"], fallback_klaster=3)
            c["klaster_id"] = k
            c["klaster_label"] = UU_6_2023_KLASTER_LABELS.get(k)
            n_penjelasan += 1
        else:
            c["jenis"] = "batang_tubuh"
            if c.get("klaster_id") is None:      # bab=None front matter -> Ketentuan Umum
                c["klaster_id"] = 1
                c["klaster_label"] = UU_6_2023_KLASTER_LABELS[1]

    return {
        "penjelasan_offset": pen_offset,
        "penjelasan_offset_pct": round(100 * pen_offset / max(len(full_text), 1), 1),
        "n_penjelasan_chunks": n_penjelasan,
        "penjelasan_routing": "keyword",
        "backbone_size": len(backbone),
        "backbone_own_pasal_max": backbone[-1][0] if backbone else 0,
        "backbone_klaster_transitions": _klaster_transitions(backbone),
    }


def _klaster_transitions(backbone: list[tuple[int, int]]) -> list[dict]:
    """Ringkas backbone jadi rentang own_pasal per klaster (utk verifikasi)."""
    out: list[dict] = []
    for own, k in backbone:
        if out and out[-1]["klaster_id"] == k:
            out[-1]["own_pasal_end"] = own
        else:
            out.append({"klaster_id": k, "own_pasal_start": own, "own_pasal_end": own})
    return out

## A.5 — Bangun Chunk dari 4 PDF

Pipeline `load → clean → chunk` dijalankan untuk keempat PDF. `UU_6_2023` mendapat
pemetaan klaster (`map_klaster_by_bab=True`) lalu di-*refine* (batang tubuh per-BAB,
Penjelasan per-keyword).

In [8]:
CS = RAG_CONFIG["chunker"]

def build_chunks_for_pdf(code_name, pdf_path):
    is_uu = code_name == "UU_6_2023"
    pages = load_pdf_pages(pdf_path, strategy="pypdf")
    full = "\n".join(clean_pages(pages))
    chunks = chunk_by_pasal(
        full, code_name,
        include_bab_metadata=True,
        chunk_size=CS["chunk_size"],
        chunk_overlap=CS["chunk_overlap"],
        hard_cap=CS["hard_cap"],
        skip_cukup_jelas=True,
        map_klaster_by_bab=is_uu,
    )
    refine = refine_uu_klaster(chunks, full) if is_uu else None
    return chunks, refine

all_chunks = {}
t0 = time.time()
for name, path in PDF_FILES.items():
    ch, refine = build_chunks_for_pdf(name, path)
    all_chunks[name] = ch
    n_sub = sum(1 for c in ch if c["is_subchunk"])
    print(f"{name:12s}: {len(ch):5d} chunks (whole={len(ch)-n_sub}, sub={n_sub})")
print(f"\nTotal {sum(len(v) for v in all_chunks.values())} chunks dalam {time.time()-t0:.1f}s")

PP_5_2021   :   749 chunks (whole=440, sub=309)
PP_35_2021  :   106 chunks (whole=70, sub=36)
PP_51_2023  :    39 chunks (whole=7, sub=32)
UU_6_2023   :  2125 chunks (whole=1359, sub=766)

Total 3019 chunks dalam 36.6s


**Bukti overlap eksplisit.** Untuk pasal yang di-sub-split, potongan berurutan berbagi
tepat 100 karakter (`chunk_overlap`). Sel berikut memverifikasi invarian
`sub[k][-100:] == sub[k+1][:100]`.

In [9]:
# Verifikasi overlap eksplisit pada sub-chunk
def verify_overlap(chunks, overlap, n=3):
    from collections import defaultdict
    groups = defaultdict(list)
    for c in chunks:
        if c["n_subchunks"] > 1:
            groups[c["chunk_id"].rsplit("_sub", 1)[0]].append(c)
    shown = 0
    for key, g in groups.items():
        g = sorted(g, key=lambda x: x["sub_index"])
        for a, b in zip(g, g[1:]):
            ok = a["text"][-overlap:] == b["text"][:overlap]
            print(f"  {a['chunk_id']} -> {b['chunk_id']}: overlap {overlap} char cocok = {ok}")
            shown += 1
            if shown >= n:
                return
verify_overlap(all_chunks["PP_35_2021"], CS["chunk_overlap"], n=3)

  PP_35_2021_pasal_1_0_sub0 -> PP_35_2021_pasal_1_0_sub1: overlap 100 char cocok = True
  PP_35_2021_pasal_1_0_sub1 -> PP_35_2021_pasal_1_0_sub2: overlap 100 char cocok = True
  PP_35_2021_pasal_1_0_sub2 -> PP_35_2021_pasal_1_0_sub3: overlap 100 char cocok = True


**Distribusi klaster UU 6/2023.** Ke-18 koleksi (3 PP + 15 klaster UU) harus terisi.
Batang tubuh dipetakan presisi per-BAB; Penjelasan di-route per-keyword.

In [10]:
# Ringkasan chunk UU per klaster + jenis
from collections import Counter
uu = all_chunks["UU_6_2023"]
kd = Counter(c.get("klaster_id") for c in uu)
jd = Counter(c.get("jenis") for c in uu)
print("Jenis:", dict(jd))
print("Char per chunk max:", max(c["char_len"] for c in uu), "(harus <= chunk_size)")
print("\nDistribusi klaster UU_6_2023:")
for k in range(1, 16):
    label = UU_6_2023_KLASTER_LABELS[k]
    print(f"  klaster {k:2d} ({label[:34]:34s}): {kd.get(k, 0)}")
assert all(kd.get(k, 0) > 0 for k in range(1, 16)), "Ada klaster kosong!"
print("\nSemua 15 klaster UU terisi.")

Jenis: {'batang_tubuh': 1442, 'penjelasan': 683}
Char per chunk max: 1200 (harus <= chunk_size)

Distribusi klaster UU_6_2023:
  klaster  1 (Ketentuan Umum                    ): 22
  klaster  2 (Asas, Tujuan, dan Ruang Lingkup   ): 4
  klaster  3 (Peningkatan Ekosistem Investasi da): 1625
  klaster  4 (Ketenagakerjaan                   ): 105
  klaster  5 (Koperasi dan UMK-M                ): 55
  klaster  6 (Kemudahan Berusaha                ): 134
  klaster  7 (Dukungan Riset dan Inovasi        ): 2
  klaster  8 (Pengadaan Tanah                   ): 51
  klaster  9 (Kawasan Ekonomi                   ): 47
  klaster 10 (Investasi Pemerintah Pusat dan Pro): 31
  klaster 11 (Administrasi Pemerintahan         ): 25
  klaster 12 (Pengawasan dan Pembinaan          ): 4
  klaster 13 (Ketentuan Lain-Lain               ): 2
  klaster 14 (Ketentuan Peralihan               ): 3
  klaster 15 (Ketentuan Penutup                 ): 15

Semua 15 klaster UU terisi.


## A.6 — Embedder BGE-M3

`BAAI/bge-m3` adalah model embedding *open-source* multilingual (mendukung Bahasa
Indonesia native, dimensi 1024, konteks hingga 8k token). Model di-*load sekali* dan
di-cache di memori — tidak di-reload per query.

In [11]:
from sentence_transformers import SentenceTransformer

EMB = RAG_CONFIG["embedder"]
embedder = SentenceTransformer(EMB["model_id"], device="cuda" if torch.cuda.is_available() else "cpu")

def embed_texts(texts):
    return embedder.encode(
        texts, batch_size=EMB["batch_size"],
        normalize_embeddings=EMB["normalize"],
        convert_to_numpy=True, show_progress_bar=False,
    )

# sanity: dimensi vektor
_dim = embed_texts(["uji dimensi embedding"]).shape[1]
print(f"Embedder {EMB['model_id']} siap. Dimensi vektor: {_dim}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedder BAAI/bge-m3 siap. Dimensi vektor: 1024


## A.7 — Vector Store: ChromaDB (18 koleksi)

Strategi **multi-koleksi**: 3 koleksi untuk PP + **15 koleksi per klaster** untuk
`UU_6_2023`. Pemisahan per-klaster membuat retrieval ter-fokus dan mendukung *metadata
filtering* pada tier berikutnya. Indeks disimpan **persisten** ke Drive.

In [12]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Nama 18 koleksi
COLLECTION_NAMES = ["pp_5_2021", "pp_35_2021", "pp_51_2023"] + \
                   [f"uu_6_2023_klaster_{k}" for k in range(1, 16)]

def collection_for(chunk):
    src = chunk["pdf_source"]
    if src == "UU_6_2023":
        return f"uu_6_2023_klaster_{chunk['klaster_id']}"
    return src.lower()   # PP_5_2021 -> pp_5_2021

def chunk_metadata(chunk):
    # ChromaDB metadata hanya menerima str/int/float/bool (tanpa None)
    return {
        "pdf_source": chunk["pdf_source"],
        "pasal": str(chunk["pasal"]),
        "bab": chunk.get("bab") or "",
        "klaster_id": int(chunk["klaster_id"]) if chunk.get("klaster_id") else 0,
        "klaster_label": chunk.get("klaster_label") or "",
        "jenis": chunk.get("jenis") or "batang_tubuh",
        "is_subchunk": bool(chunk["is_subchunk"]),
    }

# Reset koleksi lama (idempoten saat Run All ulang) lalu buat 18 koleksi kosong
for name in COLLECTION_NAMES:
    try:
        client.delete_collection(name)
    except Exception:
        pass
collections = {
    name: client.create_collection(name, metadata={"hnsw:space": RAG_CONFIG["vector_store"]["distance_metric"]})
    for name in COLLECTION_NAMES
}
print(f"{len(collections)} koleksi ChromaDB dibuat di {CHROMA_DIR}")

18 koleksi ChromaDB dibuat di /content/drive/MyDrive/PGABL/chroma_db


**Ingest.** Setiap chunk di-*embed* lalu dimasukkan ke koleksi tujuannya beserta
metadata (pdf, pasal, bab, klaster, jenis).

In [13]:
# Kelompokkan chunk per koleksi, lalu embed & masukkan per batch
from collections import defaultdict
buckets = defaultdict(list)
for name, chunks in all_chunks.items():
    for c in chunks:
        buckets[collection_for(c)].append(c)

t0 = time.time()
for cname, chunks in buckets.items():
    texts = [c["text"] for c in chunks]
    embs = embed_texts(texts)
    collections[cname].add(
        ids=[c["chunk_id"] for c in chunks],
        embeddings=[e.tolist() for e in embs],
        documents=texts,
        metadatas=[chunk_metadata(c) for c in chunks],
    )
    print(f"  {cname:24s}: {len(chunks)} chunk")
print(f"\nIngest selesai dalam {time.time()-t0:.1f}s")

  pp_5_2021               : 749 chunk
  pp_35_2021              : 106 chunk
  pp_51_2023              : 39 chunk
  uu_6_2023_klaster_1     : 22 chunk
  uu_6_2023_klaster_2     : 4 chunk
  uu_6_2023_klaster_3     : 1625 chunk
  uu_6_2023_klaster_4     : 105 chunk
  uu_6_2023_klaster_5     : 55 chunk
  uu_6_2023_klaster_6     : 134 chunk
  uu_6_2023_klaster_7     : 2 chunk
  uu_6_2023_klaster_8     : 51 chunk
  uu_6_2023_klaster_9     : 47 chunk
  uu_6_2023_klaster_10    : 31 chunk
  uu_6_2023_klaster_11    : 25 chunk
  uu_6_2023_klaster_12    : 4 chunk
  uu_6_2023_klaster_13    : 2 chunk
  uu_6_2023_klaster_14    : 3 chunk
  uu_6_2023_klaster_15    : 15 chunk

Ingest selesai dalam 107.2s


In [14]:
# Verifikasi: 18 koleksi, semua count > 0
listed = client.list_collections()
print(f"Jumlah koleksi: {len(listed)} (target 18)")
total = 0
for name in COLLECTION_NAMES:
    cnt = client.get_collection(name).count()
    total += cnt
    print(f"  {name:24s}: {cnt}")
print(f"\nTotal vektor terindeks: {total}")
assert len(listed) == 18 and all(client.get_collection(n).count() > 0 for n in COLLECTION_NAMES)
print("18 koleksi terisi.")

Jumlah koleksi: 18 (target 18)
  pp_5_2021               : 749
  pp_35_2021              : 106
  pp_51_2023              : 39
  uu_6_2023_klaster_1     : 22
  uu_6_2023_klaster_2     : 4
  uu_6_2023_klaster_3     : 1625
  uu_6_2023_klaster_4     : 105
  uu_6_2023_klaster_5     : 55
  uu_6_2023_klaster_6     : 134
  uu_6_2023_klaster_7     : 2
  uu_6_2023_klaster_8     : 51
  uu_6_2023_klaster_9     : 47
  uu_6_2023_klaster_10    : 31
  uu_6_2023_klaster_11    : 25
  uu_6_2023_klaster_12    : 4
  uu_6_2023_klaster_13    : 2
  uu_6_2023_klaster_14    : 3
  uu_6_2023_klaster_15    : 15

Total vektor terindeks: 3019
18 koleksi terisi.


## A.8 — Generator: Llama-3.2-3B fine-tuned

Generator adalah model hasil fine-tuning K1 (`PGABL-Llama-3.2-3B-GRPO`, *merged 16-bit*,
publik di Hugging Face). Dimuat dalam **4-bit** (QLoRA-style NF4) agar hemat VRAM dan muat
bersama BGE-M3 di T4. Model di-*load sekali*.

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN = RAG_CONFIG["generator"]
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
gen_tokenizer = AutoTokenizer.from_pretrained(GEN["model_id"])
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN["model_id"], quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)
gen_model.eval()
print(f"Generator {GEN['model_id']} dimuat (4-bit).")

config.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Generator nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO dimuat (4-bit).


In [16]:
@torch.no_grad()
def llm_generate(system_prompt, user_prompt, max_new_tokens=None):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    # apply_chat_template bisa mengembalikan tensor ATAU BatchEncoding (tergantung versi
    # transformers). Tangani keduanya secara robust.
    enc = gen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    if isinstance(enc, dict) or hasattr(enc, "input_ids"):
        input_ids = enc["input_ids"].to(gen_model.device)
        gen_kwargs = {"input_ids": input_ids}
        if "attention_mask" in enc:
            gen_kwargs["attention_mask"] = enc["attention_mask"].to(gen_model.device)
    else:
        input_ids = enc.to(gen_model.device)
        gen_kwargs = {"input_ids": input_ids}
    input_len = input_ids.shape[-1]
    out = gen_model.generate(
        **gen_kwargs,
        max_new_tokens=max_new_tokens or GEN["max_new_tokens"],
        do_sample=True, temperature=GEN["temperature"], top_p=GEN["top_p"],
        repetition_penalty=GEN["repetition_penalty"],
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    return gen_tokenizer.decode(
        out[0][input_len:], skip_special_tokens=True,
        clean_up_tokenization_spaces=False,   # BPE: cleanup destruktif, matikan
    ).strip()

def strip_think(text):
    # Model fine-tuned dapat menyertakan blok <think>...</think>; ambil jawaban final saja
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

## A.9 — Pipeline RAG Dasar

Alur tier **Basic**: *embed* pertanyaan → cari Top-K chunk termirip di seluruh koleksi →
susun konteks → *generate* jawaban dengan model fine-tuned.

In [17]:
def retrieve(query, k=None, collection_names=None):
    k = k or RAG_CONFIG["retriever"]["top_k"]
    q_emb = embed_texts([query])[0].tolist()
    names = collection_names or COLLECTION_NAMES
    candidates = []
    for name in names:
        col = client.get_collection(name)
        n = col.count()
        if n == 0:
            continue
        res = col.query(query_embeddings=[q_emb], n_results=min(k, n))
        for i in range(len(res["ids"][0])):
            candidates.append({
                "chunk_id": res["ids"][0][i],
                "document": res["documents"][0][i],
                "metadata": res["metadatas"][0][i],
                "distance": res["distances"][0][i],
                "collection": name,
            })
    candidates.sort(key=lambda x: x["distance"])
    return candidates[:k]

def format_sources(hits):
    return "\n\n".join(f"[{i+1}] {h['document']}" for i, h in enumerate(hits))

def rag_basic(query, k=None):
    hits = retrieve(query, k=k)
    user_prompt = RAG_CONFIG["context_template"].format(
        sources=format_sources(hits), question=query)
    raw = llm_generate(RAG_CONFIG["system_prompt"], user_prompt)
    return {"query": query, "answer": strip_think(raw), "raw": raw, "sources": hits}

### Uji coba 3 pertanyaan

Termasuk pertanyaan uji wajib **hak lembur staf admin**. Untuk tiap query ditampilkan
sumber yang di-retrieve (pdf, pasal, klaster, jarak) dan jawaban model.

In [18]:
TEST_QUERIES = [
    "Berapa upah kerja lembur untuk pekerja pada jam pertama?",
    "Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?",
    "Bagaimana formula penyesuaian upah minimum menurut regulasi terbaru?",
]

for q in TEST_QUERIES:
    out = rag_basic(q)
    print("=" * 90)
    print("TANYA :", q)
    print("-" * 90)
    for i, h in enumerate(out["sources"]):
        m = h["metadata"]
        print(f"  [{i+1}] {m['pdf_source']} Pasal {m['pasal']} "
              f"| {m['klaster_label'] or m['bab'] or '-'} | jarak={h['distance']:.3f}")
    print("-" * 90)
    print("JAWAB :", out["answer"])
    print()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


TANYA : Berapa upah kerja lembur untuk pekerja pada jam pertama?
------------------------------------------------------------------------------------------
  [1] PP_35_2021 Pasal 31 | IV | jarak=0.272
  [2] PP_35_2021 Pasal 31 | IV | jarak=0.285
  [3] PP_35_2021 Pasal 1 | I | jarak=0.312
  [4] PP_35_2021 Pasal 32 | IV | jarak=0.317
  [5] UU_6_2023 Pasal 78 | Ketenagakerjaan | jarak=0.340
------------------------------------------------------------------------------------------
JAWAB : Menurut Pasal 31 Ayat (1), Upah Kerja Lembur untuk pekerja pada jam pertama adalah 1,5 kali Upah sejam.



[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?
------------------------------------------------------------------------------------------
  [1] PP_5_2021 Pasal 176 | IV | jarak=0.339
  [2] PP_5_2021 Pasal 12 | II | jarak=0.340
  [3] PP_5_2021 Pasal 177 | IV | jarak=0.363
  [4] PP_5_2021 Pasal 15 | II | jarak=0.372
  [5] PP_5_2021 Pasal 14 | II | jarak=0.388
------------------------------------------------------------------------------------------
JAWAB : Nomor Induk Berusaha (NIB) adalah identitas unik yang diterbitkan oleh Lembaga OSS untuk setiap Pelaku Usaha. Ini mencakup data seperti profil, permodalan usaha, nomor pokok wajib pajak, KBLI, dan lokasi usaha. NIB digunakan sebagai bukti registrasi/pendaftaran Pelaku Usaha untuk melakukan kegiatan usaha dan berfungsi sebagai identitas bagi Pelaku Usaha. Selain itu, NIB juga berfungsi sebagai angka pengenal impor, hak akses kepabeanan, pendaftaran kepesertaan Pelaku Usaha untuk jaminan sosial kesehatan dan jaminan sosial 

---

**Ringkasan Bagian A (RAG Dasar / K2 Basic).** Pipeline end-to-end telah berjalan:
4 PDF → chunk per-pasal dengan overlap eksplisit → embedding BGE-M3 → 18 koleksi ChromaDB
persisten → retrieval Top-K → generasi jawaban dengan Llama-3.2-3B fine-tuned.

# Bagian B — RAG Skilled

Meningkatkan retrieval dengan empat komponen:

1. **Query routing + metadata filtering** — kata kunci pertanyaan menentukan klaster/koleksi
   kandidat (mis. *lembur* → Ketenagakerjaan). Bila tak ada kecocokan, retrieval jatuh ke
   seluruh 18 koleksi (*fallback-to-all*) agar recall terjaga.
2. **Ensemble Retriever (BM25 + Dense)** — BM25 menangkap kecocokan kata eksak (nomor pasal,
   istilah "PKWT"); *dense* (BGE-M3) menangkap kemiripan makna. Keduanya digabung dengan
   **Reciprocal Rank Fusion (RRF)**.
3. **Parent-Child Retriever** — *child* (potongan presisi) dipakai untuk mencari; konteks
   yang diberikan ke generator adalah **pasal utuh** (*parent*, hasil rekonstruksi de-overlap
   dari sub-chunk).
4. **Sitasi sumber** — setiap konteks dilabeli `[Sumber N: PDF - Klaster - Pasal]`, dan model
   diinstruksikan mengutip `[Sumber N]` pada jawabannya.

## B.1 — Layer Retriever (BM25, router, RRF, parent-store)

Modul retriever di-*inline* agar notebook mandiri.

In [19]:
# --- Retriever (BM25 + query routing + RRF + parent-child + sitasi) ---
"""
PGABL - Retriever layer (Tahap 3b Skilled).

Komponen (semua modular, portable ke domain lain):
  - tokenize + BM25Index    : sinyal sparse (exact-match: nomor pasal, istilah "PKWT")
  - route_query             : keyword -> klaster/collection kandidat (metadata filtering)
  - reciprocal_rank_fusion  : gabung ranking BM25 + Dense (RRF)
  - build_parent_store      : rekonstruksi teks pasal UTUH (parent) dari sub-chunk (child)
  - format_citation         : label sitasi [Sumber N: PDF - BAB/Klaster - Pasal]

Dense retrieval (BGE-M3 + ChromaDB) dijalankan di notebook (butuh GPU); modul ini
menyediakan logika non-model yang di-verify lokal via scripts/10_verify_skilled.py.
"""

from __future__ import annotations
import re
from collections import defaultdict
from typing import Optional


# ==================== Tokenizer untuk BM25 ====================
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text: str) -> list[str]:
    """Tokenisasi sederhana: lowercase + ambil token alfanumerik."""
    return _TOKEN_RE.findall(text.lower())


# ==================== Query routing (metadata filtering) ====================
# Keyword tema -> collection kandidat. PP dipetakan by topik:
#   PP_5_2021 = perizinan, PP_35_2021 = ketenagakerjaan, PP_51_2023 = pengupahan.
ROUTING_RULES = [
    {
        "theme": "ketenagakerjaan",
        "keywords": ["lembur", "upah", "pkwt", "pkwtt", "phk", "pemutusan hubungan kerja",
                     "pesangon", "cuti", "pekerja", "buruh", "serikat", "alih daya",
                     "outsourcing", "waktu kerja", "pengupahan", "tenaga kerja",
                     "jaminan sosial", "perjanjian kerja", "hubungan kerja"],
        "collections": ["pp_35_2021", "pp_51_2023", "uu_6_2023_klaster_4"],
    },
    {
        "theme": "perizinan_investasi",
        "keywords": ["izin", "perizinan", "nib", "oss", "kbli", "risiko", "berusaha",
                     "penanaman modal", "investasi", "lingkungan hidup", "bangunan gedung",
                     "tata ruang", "amdal"],
        "collections": ["pp_5_2021", "uu_6_2023_klaster_3", "uu_6_2023_klaster_6"],
    },
    {
        "theme": "pengupahan",
        "keywords": ["upah minimum", "ump", "umk", "umr", "formula upah",
                     "penyesuaian upah", "dewan pengupahan"],
        "collections": ["pp_51_2023", "uu_6_2023_klaster_4"],
    },
]


def route_query(query: str, all_collections: list[str]) -> tuple[list[str], Optional[str]]:
    """
    Return (collections_target, theme). Kalau ada keyword match -> collection kandidat
    tema tsb (metadata filter). Kalau tidak ada match -> semua collection (fallback-to-all).
    """
    low = query.lower()
    matched: list[str] = []
    theme = None
    for rule in ROUTING_RULES:
        if any(kw in low for kw in rule["keywords"]):
            theme = theme or rule["theme"]
            for c in rule["collections"]:
                if c in all_collections and c not in matched:
                    matched.append(c)
    if not matched:
        return list(all_collections), None
    return matched, theme


# ==================== Reciprocal Rank Fusion ====================
def reciprocal_rank_fusion(ranked_lists: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    """
    Gabung beberapa ranked list (list of ids urut relevansi) via RRF.
    Skor(id) = sum_over_lists 1 / (k + rank), rank 0-based.
    Return list (id, score) urut skor menurun.
    """
    scores: dict[str, float] = defaultdict(float)
    for lst in ranked_lists:
        for rank, item_id in enumerate(lst):
            scores[item_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# ==================== Parent-Child ====================
def parent_key_of(chunk_id: str) -> str:
    """child chunk_id -> parent_key (buang suffix _sub{j})."""
    return chunk_id.rsplit("_sub", 1)[0]


def build_parent_store(all_chunks: dict[str, list[dict]], overlap: int = 100) -> dict[str, dict]:
    """
    Bangun 'parent' = teks pasal UTUH dari sub-chunk (child). Rekonstruksi de-overlap:
    parent = sub0 + sub1[overlap:] + sub2[overlap:] + ...  (kebalikan sliding-window split,
    jadi parent == blok pasal asli). Untuk pasal 1-chunk, parent == child.

    Return {parent_key: {parent_id, text, pdf_source, pasal, bab, klaster_label, n_children}}.
    """
    groups: dict[str, list[dict]] = defaultdict(list)
    for chunks in all_chunks.values():
        for c in chunks:
            groups[parent_key_of(c["chunk_id"])].append(c)

    store: dict[str, dict] = {}
    for pkey, g in groups.items():
        g = sorted(g, key=lambda x: x["sub_index"])
        text = g[0]["text"]
        for c in g[1:]:
            text += c["text"][overlap:] if len(c["text"]) > overlap else ""
        first = g[0]
        store[pkey] = {
            "parent_id": pkey,
            "text": text,
            "pdf_source": first["pdf_source"],
            "pasal": first["pasal"],
            "bab": first.get("bab"),
            "klaster_label": first.get("klaster_label"),
            "n_children": len(g),
        }
    return store


# ==================== BM25 index ====================
class BM25Index:
    """BM25Okapi di atas seluruh chunk (global). Filter collection via allowed_ids."""

    def __init__(self, chunks: list[dict]):
        from rank_bm25 import BM25Okapi
        self.chunks = chunks
        self.ids = [c["chunk_id"] for c in chunks]
        self.bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

    def search(self, query: str, top_n: int = 20, allowed_ids: Optional[set] = None) -> list[str]:
        scores = self.bm25.get_scores(tokenize(query))
        order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        out: list[str] = []
        for i in order:
            cid = self.ids[i]
            if allowed_ids is not None and cid not in allowed_ids:
                continue
            if scores[i] <= 0:
                break
            out.append(cid)
            if len(out) >= top_n:
                break
        return out


# ==================== Citation formatting ====================
def format_citation(rank: int, meta: dict) -> str:
    """Label sitasi utk 1 sumber: [Sumber N: PDF - BAB/Klaster - Pasal X]."""
    pdf = meta.get("pdf_source", "?")
    scope = meta.get("klaster_label") or (f"BAB {meta['bab']}" if meta.get("bab") else "")
    pasal = meta.get("pasal", "?")
    scope_part = f" - {scope}" if scope else ""
    return f"[Sumber {rank}: {pdf}{scope_part} - Pasal {pasal}]"

In [20]:
from collections import defaultdict

# Index sparse (BM25) global + parent store + peta koleksi -> chunk_ids
_flat_chunks = [c for chunks in all_chunks.values() for c in chunks]
bm25 = BM25Index(_flat_chunks)
parent_store = build_parent_store(all_chunks, overlap=RAG_CONFIG["chunker"]["chunk_overlap"])

collection_ids = defaultdict(set)
for chunks in all_chunks.values():
    for c in chunks:
        collection_ids[collection_for(c)].add(c["chunk_id"])

print(f"BM25 index : {len(_flat_chunks)} chunk")
print(f"Parent store: {len(parent_store)} pasal (parent) — "
      f"{sum(1 for p in parent_store.values() if p['n_children']>1)} pasal ter-split")

BM25 index : 3019 chunk
Parent store: 2295 pasal (parent) — 419 pasal ter-split


## B.2 — Ensemble Retrieve + Parent-Child

`ensemble_retrieve` menjalankan dense (Chroma, terfilter koleksi hasil routing) dan BM25
(terfilter ke id koleksi yang sama), lalu menggabung dengan RRF. `retrieve_with_parents`
mengubah *child* hasil fusi menjadi *parent* (pasal utuh) yang unik untuk konteks generator.

In [21]:
def dense_search(query, collections, top_n=20):
    q_emb = embed_texts([query])[0].tolist()
    cand = []
    for name in collections:
        col = client.get_collection(name)
        n = col.count()
        if n == 0:
            continue
        res = col.query(query_embeddings=[q_emb], n_results=min(top_n, n))
        for i in range(len(res["ids"][0])):
            cand.append((res["ids"][0][i], res["distances"][0][i]))
    cand.sort(key=lambda x: x[1])
    return [cid for cid, _ in cand[:top_n]]

def ensemble_retrieve(query, top_n=20):
    cols, theme = route_query(query, COLLECTION_NAMES)   # metadata filtering
    allowed_ids = set().union(*(collection_ids[c] for c in cols)) if cols else set()
    dense_ranked = dense_search(query, cols, top_n=top_n)
    sparse_ranked = bm25.search(query, top_n=top_n, allowed_ids=allowed_ids)
    fused = reciprocal_rank_fusion([dense_ranked, sparse_ranked], k=60)
    return {"theme": theme, "collections": cols,
            "dense": dense_ranked, "sparse": sparse_ranked,
            "fused": [cid for cid, _ in fused]}

def retrieve_with_parents(query, top_k=5, top_n=20):
    ens = ensemble_retrieve(query, top_n=top_n)
    seen, parents = set(), []
    for cid in ens["fused"]:
        pkey = parent_key_of(cid)
        if pkey in seen:
            continue
        seen.add(pkey)
        p = dict(parent_store[pkey]); p["child_id"] = cid
        parents.append(p)
        if len(parents) >= top_k:
            break
    ens["parents"] = parents
    return ens

## B.3 — Pipeline RAG Skilled (dengan sitasi)

In [22]:
SYSTEM_PROMPT_SKILLED = (
    "Anda adalah asisten hukum untuk tim legal internal. Jawab HANYA berdasarkan konteks "
    "pasal yang diberikan. WAJIB mencantumkan sumber dengan format [Sumber N] setiap kali "
    "mengutip pasal. Gunakan Bahasa Indonesia hukum yang formal dan ringkas. Jika konteks "
    "tidak memuat jawaban, katakan: 'Informasi tidak ditemukan pada regulasi yang tersedia.'"
)
CONTEXT_TEMPLATE_SKILLED = (
    "Konteks pasal (dengan sumber):\n{sources}\n\n"
    "Pertanyaan: {question}\nJawaban (sertakan rujukan [Sumber N]):"
)

def build_cited_context(parents):
    blocks = []
    for i, p in enumerate(parents):
        blocks.append(f"{format_citation(i+1, p)}\n{p['text']}")
    return "\n\n".join(blocks)

def rag_skilled(query, top_k=None, top_n=20):
    top_k = top_k or RAG_CONFIG["retriever"]["top_k"]
    r = retrieve_with_parents(query, top_k=top_k, top_n=top_n)
    user_prompt = CONTEXT_TEMPLATE_SKILLED.format(
        sources=build_cited_context(r["parents"]), question=query)
    raw = llm_generate(SYSTEM_PROMPT_SKILLED, user_prompt)
    return {"query": query, "answer": strip_think(raw), "raw": raw,
            "parents": r["parents"], "theme": r["theme"],
            "collections": r["collections"], "dense": r["dense"], "sparse": r["sparse"]}

### Bukti Ensemble + Parent-Child

Sel berikut menampilkan perbedaan ranking Dense vs BM25 vs hasil fusi RRF, serta contoh
ekspansi *child* → *parent* (pasal panjang yang ter-split).

In [23]:
r = retrieve_with_parents("upah kerja lembur untuk jam pertama", top_k=5, top_n=15)
print("Routing theme :", r["theme"], "| koleksi:", r["collections"])
print("Dense  top5 :", r["dense"][:5])
print("BM25   top5 :", r["sparse"][:5])
print("RRF    top5 :", [p["child_id"] for p in r["parents"]])
print()
for p in r["parents"]:
    if p["n_children"] > 1:
        child = next(c for c in _flat_chunks if c["chunk_id"] == p["child_id"])
        print(f"Parent-Child: child {p['child_id']} ({len(child['text'])} char) "
              f"-> parent {p['parent_id']} ({len(p['text'])} char, {p['n_children']} sub-chunk)")
        break
else:
    print("(Top-5 kali ini pasal pendek — child = parent)")

Routing theme : ketenagakerjaan | koleksi: ['pp_35_2021', 'pp_51_2023', 'uu_6_2023_klaster_4']
Dense  top5 : ['PP_35_2021_pasal_31_30_sub0', 'PP_35_2021_pasal_31_30_sub1', 'PP_35_2021_pasal_32_31', 'PP_35_2021_pasal_1_0_sub1', 'PP_35_2021_pasal_26_26']
BM25   top5 : ['PP_35_2021_pasal_31_30_sub0', 'PP_35_2021_pasal_31_30_sub1', 'UU_6_2023_pasal_78_819', 'PP_35_2021_pasal_34_34', 'PP_35_2021_pasal_26_26']
RRF    top5 : ['PP_35_2021_pasal_31_30_sub0', 'UU_6_2023_pasal_78_819', 'PP_35_2021_pasal_32_31', 'PP_35_2021_pasal_26_26', 'PP_35_2021_pasal_1_0_sub1']

Parent-Child: child PP_35_2021_pasal_31_30_sub0 (1200 char) -> parent PP_35_2021_pasal_31_30 (1853 char, 2 sub-chunk)


### Uji coba dengan sitasi

Pertanyaan dari domain legal; jawaban wajib memuat rujukan `[Sumber N]`.

In [24]:
import re as _re

SKILLED_TESTS = [
    "Berapa upah kerja lembur untuk 1 jam pertama bagi staf admin?",
    "Berapa besar uang kompensasi bagi pekerja dengan PKWT?",
    "Apa yang dimaksud Perizinan Berusaha Berbasis Risiko?",
]
for q in SKILLED_TESTS:
    out = rag_skilled(q)
    print("=" * 90)
    print("TANYA :", q)
    print(f"  routing = {out['theme'] or 'ALL'} | {len(out['collections'])} koleksi")
    print("-" * 90)
    for i, p in enumerate(out["parents"]):
        print(f"  [Sumber {i+1}] {p['pdf_source']} Pasal {p['pasal']} "
              f"| {p['klaster_label'] or (('BAB ' + p['bab']) if p['bab'] else '-')}")
    print("-" * 90)
    print("JAWAB :", out["answer"])
    print("Memuat sitasi [Sumber N]:", bool(_re.search(r"\[Sumber\s*\d+", out["answer"])))
    print()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Berapa upah kerja lembur untuk 1 jam pertama bagi staf admin?
  routing = ketenagakerjaan | 3 koleksi
------------------------------------------------------------------------------------------
  [Sumber 1] PP_35_2021 Pasal 31 | BAB IV
  [Sumber 2] UU_6_2023 Pasal 78 | Ketenagakerjaan
  [Sumber 3] PP_35_2021 Pasal 1 | BAB I
  [Sumber 4] PP_35_2021 Pasal 26 | BAB IV
  [Sumber 5] PP_35_2021 Pasal 28 | BAB IV
------------------------------------------------------------------------------------------
JAWAB : Maaf, saya tidak bisa menemukan informasi tentang upah kerja lembur untuk 1 jam pertama bagi staf admin dalam teks yang diberikan. Apakah Anda ingin memberikan konteks atau sumber yang lebih spesifik?
Memuat sitasi [Sumber N]: False



[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Berapa besar uang kompensasi bagi pekerja dengan PKWT?
  routing = ketenagakerjaan | 3 koleksi
------------------------------------------------------------------------------------------
  [Sumber 1] PP_35_2021 Pasal 15 | BAB II
  [Sumber 2] PP_35_2021 Pasal 16 | BAB II
  [Sumber 3] PP_35_2021 Pasal 17 | BAB II
  [Sumber 4] PP_35_2021 Pasal 66 | BAB IX
  [Sumber 5] PP_35_2021 Pasal 64 | BAB IX
------------------------------------------------------------------------------------------
JAWAB : Berapa besar uang kompensasi bagi pekerja dengan PKWT? 

Menurut Pasal 64 Peraturan Pemerintah Republik Indonesia Nomor 35 Tahun 2021, besaran uang kompensasi sebagaimana dimaksud pada huruf a dihitung berdasarkan masa kerja pekerja yang perhitungannya dimulai sejak tanggal diundangkan Undang-Undang Nomor 11 Tahun 2020 tentang Cipta Kerja.
Memuat sitasi [Sumber N]: False

TANYA : Apa yang dimaksud Perizinan Berusaha Berbasis Risiko?
  routing = perizinan_investasi | 3 koleksi
----------------

---

**Ringkasan Bagian B (RAG Skilled / K2 Skilled).** Retrieval kini memakai *metadata
filtering* via query routing, *ensemble* BM25+Dense (RRF), dan *parent-child* (child presisi
→ parent pasal utuh), dengan jawaban ber-*sitasi* `[Sumber N]`. Bagian berikutnya menambahkan
HyDE, reranker, ambang relevansi, dan *fallback* pencarian web.